In [ ]:
import xarray as xr
from dask.distributed import Client, LocalCluster
import os
import glob as glob
import glide.science_data_processing.L1A as L1A

imager = "WFI"
# 1. Set environment variables to prevent underlying library thread conflicts
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

# 2. Configure a local cluster optimized for 128 GB RAM and 64 cores
# We restrict to 16 workers with 4 threads each to minimize HDF5 lock contention
cluster = LocalCluster(
    n_workers=16,          # 16 separate Python processes
    threads_per_worker=4,  # 4 threads per process (64 total threads used)
    memory_limit="7.5GB",  # 16 * 7.5GB = 120GB (leaves 8GB safety buffer for OS)
)

# 3. Connect Dask to this custom infrastructure
client = Client(cluster)

# Print the dashboard URL so you can monitor progress live
print(f"Dask Dashboard is live at: {client.dashboard_link}")


file_paths = sorted(glob.glob(f"/data/L1A/CARRUTHERS_GCI-{imager}_L1A-DRK_202601[1-3][0-9]_v1.0.nc"))
print(file_paths)

ds_all = xr.open_mfdataset(
    file_paths, 
    combine="nested",
    engine='netcdf4', 
    concat_dim="time", 
    chunks={"time": 10},  # Lazy loading using Dask (adjust chunk size as needed)
    parallel=True         # Speeds up parsing metadata across cores
)

# 3. Access your L1A wrapper
l1a_all = L1A.L1A(ds_all)

# 3. Now ds_all is a single xarray Dataset with all your data combined along 'time'
# Save dataset
ds_all.to_netcdf(f"products/CARRUTHERS_GCI-{imager}_L1A-DRK_combined_v1.0.nc", mode="w")